# ◆ Studio IA — Estudio Generativo en Google Colab

1. Arriba a la derecha en Colab, elige la tarjeta gráfica que quieras usar hoy (**T4, L4, A100...**) en `Entorno de ejecución → Cambiar tipo de entorno de ejecución`.
2. Pulsa el botón **Play (▶)** de la celda de abajo.
3. Abre el enlace `https://...trycloudflare.com` que aparecerá en unos segundos.

In [ ]:
import os, subprocess, importlib
branch = 'arena/f0997b49-studio-ai'
repo = '/content/studio'
if not os.path.exists(repo):
    subprocess.check_call(['git', 'clone', '-b', branch, '--single-branch', 'https://github.com/zurdonman/Studio-ai.git', repo])
else:
    current = subprocess.check_output(['git', '-C', repo, 'branch', '--show-current'], text=True).strip()
    if current != branch:
        raise RuntimeError(f'La carpeta está en {current}, no en {branch}. No se cambian ramas ni se borran datos automáticamente.')
    subprocess.check_call(['git', '-C', repo, 'pull', '--ff-only', 'origin', branch])
os.chdir(repo)
print('Revisión:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
from studio import diagnostics
importlib.reload(diagnostics)
import launch_colab
importlib.reload(launch_colab)
launch_colab.main()


## Rescatar el diagnóstico aunque la web no abra
Si falla el enlace, detén la celda anterior si sigue bloqueada y ejecuta esta celda. No reinicies ni elimines el entorno: primero descarga los logs y tus resultados. El ZIP no incluye pesos ni archivos multimedia y oculta tokens. Solo funciona mientras el disco de esta VM siga existiendo; `/content` no persiste si Colab elimina la máquina.


In [ ]:
import os, sys, subprocess
from pathlib import Path
repo = Path('/content/studio')
if not (repo / 'studio' / 'diagnostics.py').exists():
    raise RuntimeError('Este checkout aún no contiene el recolector. Conserva /content/studio_data y actualiza el código sin reiniciar la VM.')
zip_path = subprocess.check_output([sys.executable, '-m', 'studio.diagnostics', '--data-dir', os.environ.get('STUDIO_DATA_DIR', '/content/studio_data')], cwd=str(repo), text=True).strip()
print('Diagnóstico:', zip_path)
from google.colab import files
files.download(zip_path)
